# Basic Swaps

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter
from io import BytesIO

import torch
from IPython.display import Image, Markdown, SVG, display
from matplotlib import pyplot as plt
from sympy import Symbol

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import CliffordCall, ControlledSwapCall
from flagquantum.compiler.registers.primitives.controlled_swap import controlled_swap_bit_ir, controlled_swap_ir
from flagquantum.compiler.registers.primitives.swap import source_swap_resource_formula, swap_register_ir
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.simulation.unitary import get_unitary

dtype = torch.complex128

def show_graph(name, child, count, leaf_name):
    graph = symbolic_call_graph(name, {name: {child: count}}, max_depth=1)
    def title(node):
        return leaf_name if node == child else name
    display(SVG(render_call_graph_svg(export_call_graph_dot(name, graph.direct_calls, title=title))))
    display(Markdown(format_call_graph_markdown(name, graph.direct_calls, graph.leaf_counts, title=title)))

## `TwoBitSwap`
Swap two bits.

This is a Clifford operation.

#### Registers
 - `x`: the first bit
 - `y`: the second bit


In [ ]:
swap_leaf = CliffordCall("swap_bit", 2)

### Example Instances

In [ ]:
swap_bit = Signature.build(x=1, y=1)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("TwoBitSwap", swap_bit, {})])))

## `TwoBitCSwap`
Swap two bits controlled on a control bit.

This is sometimes known as the [Fredkin Gate](https://en.wikipedia.org/wiki/Fredkin_gate).

#### Registers
 - `ctrl`: the control bit
 - `x`: the first bit
 - `y`: the second bit 

#### References
 - [An algorithm for the T-count](https://arxiv.org/abs/1308.4134). Gosset et al. 2013. Figure 5.2.


In [ ]:
cswap_leaf = ControlledSwapCall(1)

### Example Instances

In [ ]:
cswap_bit = Signature.build(ctrl=1, x=1, y=1)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("TwoBitCSwap", cswap_bit, {})])))

### Clifford+T circuit

In Qualtran, this bloq is treated as atomic because its implementation is architecture-dependent. A clifford+T compilation is provided via a Cirq circuit on the `to_clifford_t_circuit()` method

The native drawing uses the same exact Clifford+T primitive sequence. Its emitted counts are separate from the declared cost of the atomic call.

In [ ]:
gate_flow = controlled_swap_bit_ir(0, 1, 2)
fredkin = Circuit(3, dtype=dtype).append(FieldOperation(3, gate_flow))
fig, _ = fredkin.draw("mpl")
with BytesIO() as png:
    fig.savefig(png, format="png", bbox_inches="tight")
    display(Image(data=png.getvalue()))
plt.close(fig)
print(dict(Counter(g.name for g in gate_flow)))

## `Swap`
Swap two registers

This corresponds to a qubitwise `TwoBitSwap` on the two registers.

#### Parameters
 - `bitsize`: The bitsize of each of the two registers being swapped. 

#### Registers
 - `x`: the first register
 - `y`: the second register


In [ ]:
def swap_signature(width):
    return Signature.build(x=width, y=width)

### Example Instances

In [ ]:
n = Symbol("n", positive=True, integer=True)
swap = swap_signature(n)

In [ ]:
swap_small = swap_signature(4)

In [ ]:
swap_large = swap_signature(64)

#### Graphical Signature

In [ ]:
examples = (("Swap(n)", swap), ("Swap(4)", swap_small), ("Swap(64)", swap_large))
display(SVG(field_signature_svg([(name, sig, {}) for name, sig in examples])))

### Call Graph

In [ ]:
show_graph("Swap(n)", swap_leaf, n, "TwoBitSwap")
print(source_swap_resource_formula(n))

## `CSwap`
Swap two registers controlled on a control bit.

This decomposes into a qubitwise `TwoBitCSwap` on the two target registers,
and takes $n$ TwoBitCSwap gates.

#### Parameters
 - `bitsize`: The bitsize of each of the two registers being swapped. 

#### Registers
 - `ctrl`: the control bit
 - `x`: the first register
 - `y`: the second register


In [ ]:
def cswap_signature(width):
    return Signature.build(ctrl=1, x=width, y=width)

### Example Instances

In [ ]:
cswap = cswap_signature(n)

In [ ]:
cswap_small = cswap_signature(4)

In [ ]:
cswap_large = cswap_signature(64)

#### Graphical Signature

In [ ]:
examples = (("CSwap(n)", cswap), ("CSwap(4)", cswap_small), ("CSwap(64)", cswap_large))
display(SVG(field_signature_svg([(name, sig, {}) for name, sig in examples])))

### Call Graph

In [ ]:
show_graph("CSwap(n)", cswap_leaf, n, "TwoBitCSwap")
print(query_qec_gate_cost(ControlledSwapCall(n)))

### Append exact register exchanges

Bind each pair in most-significant-bit order. Compare the complete complex matrices for a two-bit register pair. Verify both control sectors and the self-adjoint inverse. The 4-bit and 64-bit examples above describe register shapes without building a dense state for the larger example.

In [ ]:
left, right = (1, 2), (3, 4)
for controlled in (False, True):
    width = 5 if controlled else 4
    x, y = (left, right) if controlled else ((0, 1), (2, 3))
    gates = controlled_swap_ir(0, x, y) if controlled else swap_register_ir(x, y)
    circuit = Circuit(width, dtype=dtype).append(FieldOperation(width, gates))
    permutation = []
    for column in range(1 << width):
        high, low = (column >> 2) & 3, column & 3
        active = not controlled or column >= 16
        out = (low << 2) | high if active else column & 15
        permutation.append((column & 16) | out)
    expected = torch.zeros((1 << width, 1 << width), dtype=dtype)
    expected[permutation, torch.arange(1 << width)] = 1
    torch.testing.assert_close(get_unitary(circuit), expected, atol=2e-12, rtol=0)
    circuit.append(FieldOperation(width, gates))
    torch.testing.assert_close(get_unitary(circuit), torch.eye(1 << width, dtype=dtype), atol=2e-12, rtol=0)
    print("Controlled" if controlled else "Unconditional", dict(Counter(g.name for g in gates)))